<a href="https://colab.research.google.com/github/IanHollow/cs5782-final-project/blob/main/notebooks/colab_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DoRA Colab Runner for a Private GitHub Repo

This notebook is the recommended way to run this project in Google Colab when the repository is private.

Recommended setup:
- Store this notebook in Colab or Google Drive, not as the primary launch point inside the private GitHub repo.
- Put `GITHUB_TOKEN`, `GITHUB_USERNAME`, and `HF_TOKEN` in Colab Secrets.
- Mount Google Drive so checkpoints, caches, and results survive runtime resets.
- Clone the private repo at runtime with an `askpass` helper so the token is not embedded into the git remote URL.

Why this flow:
- The current Open in Colab GitHub flow is just a URL redirect to `colab.research.google.com/github/...`, which is naturally aligned with notebooks hosted on GitHub pages, especially public ones.
- Colab runtimes are ephemeral, so Drive-backed workspaces are the safest default for long fine-tuning jobs.
- Hugging Face tokens are the standard way to access gated Llama-family models from notebooks.

## Before You Run

1. Start in a CPU runtime if you want to pre-download models and datasets to Drive before spending GPU time. Switch to a GPU runtime only when you are ready to train.
2. Open the Secrets pane and create:
   - `GITHUB_TOKEN`: a GitHub personal access token with access to this private repo
   - `GITHUB_USERNAME`: your GitHub username
   - `HF_TOKEN`: your Hugging Face token for gated model access
3. Make sure your Hugging Face account has accepted the model license for `meta-llama/Llama-2-7b-hf` or `meta-llama/Meta-Llama-3-8B` if you want the full experiments.
4. If you do not want to grant GitHub access in Colab, use the ZIP upload mode below and upload a repo snapshot instead.

In [15]:
from __future__ import annotations

import os
import shutil
import subprocess  # noqa: S404
import sys
import tempfile
from pathlib import Path
from typing import Never

try:
    from google.colab import drive, files, userdata  # ty:ignore[unresolved-import]
except ImportError as exc:
    message = "This notebook is meant to run inside Google Colab."
    raise RuntimeError(message) from exc

# @title Colab Parameters
REPO_FULL_NAME = "IanHollow/cs5782-final-project"
REPO_BRANCH = "main"
SOURCE_MODE = "git_clone"
WORKSPACE_ROOT = "/content/drive/MyDrive/cs5782_dora/final-project"
REPO_DIR_NAME = "."
PERSIST_HF_CACHE = True


def fail(message: str) -> Never:
    raise RuntimeError(message)


def sh(cmd: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> None:
    print("+", " ".join(cmd))
    process = subprocess.Popen(  # noqa: S603
        cmd,
        cwd=None if cwd is None else str(cwd),
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    if process.stdout is None:
        fail("Subprocess stdout pipe was not created.")
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)


def capture(cmd: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> str:
    print("+", " ".join(cmd))
    completed = subprocess.run(  # noqa: S603
        cmd,
        check=True,
        cwd=None if cwd is None else str(cwd),
        env=env,
        text=True,
        capture_output=True,
    )
    return completed.stdout.strip()


def get_secret(name: str, *, required: bool = False) -> str | None:
    env_value = os.environ.get(name)
    if env_value:
        return env_value

    try:
        value = userdata.get(name)
    except (
        userdata.NotebookAccessError,
        userdata.SecretNotFoundError,
        userdata.TimeoutException,
        ValueError,
    ):
        value = None

    if value:
        return value
    if required:
        fail(f"Missing required Colab secret: {name}")
    return None


WORKSPACE_ROOT = Path(WORKSPACE_ROOT)
REPO_DIR = WORKSPACE_ROOT / REPO_DIR_NAME
DRIVE_MOUNT = Path("/content/drive")

In [16]:
# @title Mount Drive and Prepare Directories
if not DRIVE_MOUNT.exists() or not any(DRIVE_MOUNT.iterdir()):
    drive.mount(str(DRIVE_MOUNT))

REPO_DIR.mkdir(parents=True, exist_ok=True)
(REPO_DIR / "artifacts").mkdir(parents=True, exist_ok=True)
print(f"Workspace root: {REPO_DIR}")

Workspace root: /content/drive/MyDrive/cs5782_dora/final-project


In [17]:
def clone_private_repo(repo_full_name: str, branch: str, repo_dir: Path) -> None:
    github_token = get_secret("GITHUB_TOKEN", required=True)
    if github_token is None:
        fail("Set GITHUB_TOKEN in Colab Secrets before cloning the private repo.")

    github_username = get_secret("GITHUB_USERNAME")
    if github_username is None:
        fail("Set GITHUB_USERNAME in Colab Secrets or in the fallback form field.")

    askpass_path = Path(tempfile.mkdtemp(prefix="git-askpass-")) / "git_askpass.sh"
    askpass_path.write_text(
        "#!/bin/sh\n"
        'case "$1" in\n'
        '  *Username*) echo "$GITHUB_USERNAME" ;;\n'
        '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
        "esac\n",
        encoding="utf-8",
    )
    askpass_path.chmod(0o700)

    askpass_env: dict[str, str] = {
        "GIT_ASKPASS": str(askpass_path),
        "GIT_TERMINAL_PROMPT": "0",
        "GITHUB_TOKEN": github_token,
        "GITHUB_USERNAME": github_username,
    }
    env = os.environ.copy()
    env.update(askpass_env)

    repo_url = f"https://github.com/{repo_full_name}.git"
    if repo_dir.exists() and not (repo_dir / ".git").exists():
        shutil.rmtree(repo_dir)

    if repo_dir.exists():
        sh(["git", "fetch", "origin", branch], cwd=repo_dir, env=env)
        sh(["git", "checkout", branch], cwd=repo_dir, env=env)
        sh(["git", "pull", "--ff-only", "origin", branch], cwd=repo_dir, env=env)
    else:
        sh(["git", "clone", "--branch", branch, repo_url, str(repo_dir)], env=env)


if SOURCE_MODE == "git_clone":
    clone_private_repo(REPO_FULL_NAME, REPO_BRANCH, REPO_DIR)
else:
    print("Upload a zip archive of the repo root.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        fail("Upload exactly one .zip file.")
    archive_name = next(iter(uploaded))
    upload_path = Path("/content") / archive_name
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    shutil.unpack_archive(str(upload_path), str(REPO_DIR.parent))
    extracted_dirs = [path for path in REPO_DIR.parent.iterdir() if path.is_dir()]
    if REPO_DIR not in extracted_dirs:
        candidates = sorted(extracted_dirs, key=lambda item: item.stat().st_mtime, reverse=True)
        if not candidates:
            fail("Could not find extracted repo directory.")
        if REPO_DIR.exists():
            shutil.rmtree(REPO_DIR)
        candidates[0].rename(REPO_DIR)

print(f"Repo dir: {REPO_DIR}")
print(capture(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR))

+ git fetch origin main
From https://github.com/IanHollow/cs5782-final-project
 * branch            main       -> FETCH_HEAD
+ git checkout main
Already on 'main'
M	.envrc
M	notebooks/bootstrap_colab.sh
M	run_docker_tests.sh
M	run_tests.sh
Your branch is up to date with 'origin/main'.
+ git pull --ff-only origin main
From https://github.com/IanHollow/cs5782-final-project
 * branch            main       -> FETCH_HEAD
Already up to date.
Repo dir: /content/drive/MyDrive/cs5782_dora/final-project
+ git rev-parse --short HEAD
39e5465


In [18]:
# @title Install the Project into the Colab Runtime
if PERSIST_HF_CACHE:
    hf_home = REPO_DIR / ".hf_home"
    hf_home.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(hf_home)
    os.environ["HF_HUB_CACHE"] = str(hf_home / "hub")

hf_token = get_secret("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

pip_cmd = [sys.executable, "-m", "pip"]
gpu_runtime = shutil.which("nvidia-smi") is not None
editable_target = ".[gpu]" if gpu_runtime else "."
sh([*pip_cmd, "install", "--editable", editable_target], cwd=REPO_DIR, env=os.environ.copy())
sh([sys.executable, "--version"], cwd=REPO_DIR)

+ /usr/bin/python3 -m pip install --editable .[gpu]
Obtaining file:///content/drive/MyDrive/cs5782_dora/final-project
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for dora-repro (pyproject.toml): started
  Building editable for dora-repro (pyproject.toml): finished with status 'done'
  Created wheel for dora-repro: filename=dora_repro-0.1.0-py3-none-any.whl size=4127 sha256=f7c68af84cdcf85ffbe9bf4f0d8b4c3ea82739a886003152333522ad7f5043ba
  Stored in directory: /tmp/pip-ephem-wheel-cache-lx5msxdq/wheels/c9/81/ed/d837b0135f13

In [19]:
# @title Smoke-Test the Repo Locally in Colab
sh(
    [
        sys.executable,
        "-m",
        "dora_repro.cli",
        "smoke-test",
        "--output-dir",
        "results/smoke-colab",
    ],
    cwd=REPO_DIR,
)

+ /usr/bin/python3 -m dora_repro.cli smoke-test --output-dir results/smoke-colab
[23:15:52] INFO     Running smoke test | command=smoke-test                     
                    output_dir=/content/drive/MyDrive/cs5782_dora/final-project/
                    results/smoke-colab                                         
           INFO     Smoke test completed |                                      
                    output_dir=/content/drive/MyDrive/cs5782_dora/final-project/
                    results/smoke-colab                                         
                    output_path=/content/drive/MyDrive/cs5782_dora/final-project
                    /results/smoke-colab/smoke_test.json                        
           INFO     Smoke test completed | command=smoke-test                   
                    output_dir=/content/drive/MyDrive/cs5782_dora/final-project/
                    results/smoke-colab                                         
                    output_p

## Runtime Selection Guidance

Use these presets by GPU type:
- `colab_t4_llama`: T4 or other smaller Colab GPUs
- `colab_l4_llama`: L4, recommended default for the full 7B/8B runs
- `colab_a100_40gb_llama`: A100 40GB, fastest sensible option for same-day turnaround
- `colab_a100_80gb_llama`: A100 80GB, only if Colab gives you the larger A100 and you want maximum throughput
- `official`: CPU-only smoke tests or debugging, not the real 7B/8B experiments

For a first end-to-end check in Colab, start with `tiny_debug` plus the `debug_quick` experiment. Once the notebook flow is stable and your `HF_TOKEN` works, switch to `llama2_7b` or `llama3_8b` with the `paper_colab` experiment.

Both `DoRA` and `LoRA` are supported with the same scope options: `full`, `attention_only`, and `mlp_only`. If you want the notebook to run from environment variables instead of the visible widget values, set `DORA_REPRO_MODEL`, `DORA_REPRO_METHOD`, `DORA_REPRO_SCOPE`, `DORA_REPRO_RUNTIME`, `DORA_REPRO_EXPERIMENT`, `DORA_REPRO_TRAIN_DATA_PATH`, `DORA_REPRO_RUN_NAME`, and `DORA_REPRO_EVAL_TASKS` before executing the training and evaluation cells.

If you want to avoid wasting GPU time on downloads, run the asset-prep cell above while attached to a CPU runtime. It stores benchmark JSONL files under `data/benchmarks/`, and model weights go into the standard Hugging Face cache on Drive because this notebook sets `HF_HOME` and `HF_HUB_CACHE` there.

In [20]:
# @title Inspect the GPU and Pick a Runtime Preset
DEFAULT_RUNTIME = "auto"  # @param ["auto", "official", "colab_t4_llama", "colab_l4_llama", "colab_a100_40gb_llama"]


def detect_gpu_name() -> str:
    if not shutil.which("nvidia-smi"):
        return ""
    try:
        return (
            capture(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"])
            .splitlines()[0]
            .strip()
        )
    except (IndexError, subprocess.CalledProcessError):
        return ""


def detect_gpu_memory_mib() -> int:
    if not shutil.which("nvidia-smi"):
        return 0
    try:
        return int(
            capture(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"])
            .splitlines()[0]
            .strip()
        )
    except (IndexError, ValueError, subprocess.CalledProcessError):
        return 0


gpu_name = detect_gpu_name()
gpu_memory_mib = detect_gpu_memory_mib()
print("GPU:", gpu_name or "CPU/MPS-like environment")
if gpu_memory_mib:
    print("GPU memory (MiB):", gpu_memory_mib)
env_runtime = os.environ.get("DORA_REPRO_RUNTIME", "").strip()
if env_runtime:
    runtime_preset = env_runtime
elif DEFAULT_RUNTIME != "auto":
    runtime_preset = DEFAULT_RUNTIME
elif "A100" in gpu_name:
    runtime_preset = "colab_a100_40gb_llama"
elif "L4" in gpu_name:
    runtime_preset = "colab_l4_llama"
elif gpu_name:
    runtime_preset = "colab_t4_llama"
else:
    runtime_preset = "official"

print("Chosen runtime preset:", runtime_preset)

+ nvidia-smi --query-gpu=name --format=csv,noheader
+ nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits
GPU: NVIDIA A100-SXM4-80GB
GPU memory (MiB): 81920
Chosen runtime preset: colab_a100_40gb_llama


In [21]:
# @title Training Configuration
MODEL_NAME = "llama2_7b"  # @param ["tiny_debug", "llama2_7b", "llama3_8b"]
METHOD = "lora"  # @param ["dora", "lora"]
SCOPE = "full"  # @param ["full", "attention_only", "mlp_only"]
EXPERIMENT = "paper_colab"  # @param ["debug_quick", "default", "paper_llama2_7b", "paper_colab"]
RANK = "8"  # @param {type:"string"}
RUN_NAME = "lora_full_rank_8"  # @param {type:"string"}
TRAIN_DATA_PATH = "/content/drive/MyDrive/cs5782_dora/final-project/data/commonsense_15k.json"  # @param {type:"string"}
RESUME_IF_POSSIBLE = False  # @param {type:"boolean"}


def env_override(name: str, current: str) -> str:
    env_value = os.environ.get(f"DORA_REPRO_{name}", "").strip()
    return env_value or current


MODEL_NAME = env_override("MODEL", MODEL_NAME)
METHOD = env_override("METHOD", METHOD)
SCOPE = env_override("SCOPE", SCOPE)
EXPERIMENT = env_override("EXPERIMENT", EXPERIMENT)
RANK = env_override("RANK", RANK)
RUN_NAME = env_override("RUN_NAME", RUN_NAME)
TRAIN_DATA_PATH = env_override("TRAIN_DATA_PATH", TRAIN_DATA_PATH)


def latest_checkpoint(run_dir: Path) -> Path | None:
    checkpoint_root = run_dir / "checkpoints"
    if not checkpoint_root.exists():
        return None
    candidates = sorted(
        checkpoint_root.glob("checkpoint-*"),
        key=lambda item: int(item.name.split("-")[-1]),
    )
    return candidates[-1] if candidates else None


resolved_run_name = RUN_NAME or f"colab-{MODEL_NAME}-{METHOD}-{SCOPE}"
run_dir = REPO_DIR / "results" / "runs" / resolved_run_name
resume_checkpoint = latest_checkpoint(run_dir) if RESUME_IF_POSSIBLE else None
print("Run dir:", run_dir)
print("Resume checkpoint:", resume_checkpoint)

Run dir: /content/drive/MyDrive/cs5782_dora/final-project/results/runs/lora_full_rank_8
Resume checkpoint: None


In [22]:
# @title Launch Training
if MODEL_NAME in {"llama2_7b", "llama3_8b"} and not os.environ.get("HF_TOKEN"):
    fail("Set HF_TOKEN in Colab Secrets before launching a gated Llama run.")

train_cmd = [
    sys.executable,
    "-m",
    "dora_repro.cli",
    "train",
    "--model",
    MODEL_NAME,
    "--method",
    METHOD,
    "--scope",
    SCOPE,
    "--runtime",
    runtime_preset,
    "--experiment",
    EXPERIMENT,
    "--output-dir",
    "results/runs",
    "--run-name",
    resolved_run_name,
]
if TRAIN_DATA_PATH:
    train_cmd.extend(["--train-data-path", TRAIN_DATA_PATH])
if RANK.strip():
    train_cmd.extend(["--rank", RANK.strip()])
if resume_checkpoint is not None:
    train_cmd += ["--resume-from-checkpoint", str(resume_checkpoint)]

sh(train_cmd, cwd=REPO_DIR, env=os.environ.copy())

+ /usr/bin/python3 -m dora_repro.cli train --model llama2_7b --method lora --scope full --runtime colab_a100_40gb_llama --experiment paper_colab --output-dir results/runs --run-name lora_full_rank_8 --train-data-path /content/drive/MyDrive/cs5782_dora/final-project/data/ --rank 8
[23:16:05] INFO     Building experiment spec | command=train method=lora        
                    model=llama2_7b run_name=lora_full_rank_8                   
                    runtime=colab_a100_40gb_llama scope=full                    
           INFO     Starting training run | command=train method=lora           
                    model=llama2_7b                                             
                    run_dir=/content/drive/MyDrive/cs5782_dora/final-project/res
                    ults/runs/lora_full_rank_8 run_name=lora_full_rank_8        
                    runtime=colab_a100_40gb_llama scope=full                    
           INFO     Writing run snapshot | method=lora model=llama2_7b 

CalledProcessError: Command '['/usr/bin/python3', '-m', 'dora_repro.cli', 'train', '--model', 'llama2_7b', '--method', 'lora', '--scope', 'full', '--runtime', 'colab_a100_40gb_llama', '--experiment', 'paper_colab', '--output-dir', 'results/runs', '--run-name', 'lora_full_rank_8', '--train-data-path', '/content/drive/MyDrive/cs5782_dora/final-project/data/', '--rank', '8']' returned non-zero exit status 1.

In [ ]:
# @title Evaluate One Run
EVAL_TASKS = "boolq,piqa,social_i_qa,hellaswag,winogrande,ARC-Easy,ARC-Challenge,openbookqa"  # @param {type:"string"}
EVAL_TASKS = env_override("EVAL_TASKS", EVAL_TASKS)

eval_cmd = [
    sys.executable,
    "-m",
    "dora_repro.cli",
    "evaluate",
    "--run-dir",
    f"results/runs/{resolved_run_name}",
]
if EVAL_TASKS.strip().lower() != "all":
    eval_cmd += ["--tasks", *[item.strip() for item in EVAL_TASKS.split(",") if item.strip()]]

sh(eval_cmd, cwd=REPO_DIR, env=os.environ.copy())

In [ ]:
# @title Summarize Runs and Inspect Outputs
sh(
    [
        sys.executable,
        "-m",
        "dora_repro.cli",
        "summarize",
        "--results-dir",
        "results/runs",
        "--output-dir",
        "results/summary",
    ],
    cwd=REPO_DIR,
)

summary_csv = REPO_DIR / "results" / "summary" / "summary.csv"
print(summary_csv)
print(summary_csv.read_text(encoding="utf-8"))

## Notes for the Real Course Runs

- Keep `results/` and the HF cache on Drive so you do not lose outputs when the runtime resets.
- Start with one run per notebook session instead of trying to multiplex many training jobs.
- For the paper-scale runs, use `llama2_7b` or `llama3_8b`, set `runtime_preset` from the detected GPU, and keep `RESUME_IF_POSSIBLE = True`.
- If private GitHub auth is annoying for collaborators, the practical fallback is to upload a zip snapshot of the repo root and run the notebook in `upload_zip` mode.